# C00 - Base canonica de audio

Este notebook deja una base comun para los preprocesamientos del Proyecto 2. Sigue la idea del Taller 1: separar lo que es canonico del dataset de las decisiones que aprende cada pipeline.

No escala, no imputa, no entrena y no crea features nuevas. Solo fija: splits disponibles, columnas de labels, archivos problematicos, conteos de etiquetas y artefactos de resumen en `02_preprocesamiento/results/`.


## Imports y configuracion


In [1]:
from __future__ import annotations

from pathlib import Path
import json
import sys
import zipfile

import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ModuleNotFoundError:
    display = print

ROOT = Path.cwd()
if ROOT.name == '02_preprocesamiento':
    ROOT = ROOT.parent
DATA_DIR = ROOT / 'data'
RESULTS_DIR = ROOT / '02_preprocesamiento' / 'results'
INVESTIGATION = ROOT / 'investigation'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(INVESTIGATION))

pd.set_option('display.max_columns', 160)
pd.set_option('display.width', 220)

from scripts.fat2019.data import CORRUPT_OR_BAD_LABEL_FILES, split_labels
from scripts.fat2019.features import read_wav_mono, log_mel_spectrogram, extract_log_mel_stats

curated = pd.read_csv(DATA_DIR / 'train_curated.csv')
noisy = pd.read_csv(DATA_DIR / 'train_noisy.csv')
sample = pd.read_csv(DATA_DIR / 'sample_submission.csv')
label_columns = list(sample.columns[1:])

print(f'ROOT={ROOT}')
print(f'DATA_DIR={DATA_DIR}')
print(f'RESULTS_DIR={RESULTS_DIR}')


ROOT=/home/santig14/fing/taa/2_TallerAA
DATA_DIR=/home/santig14/fing/taa/2_TallerAA/data
RESULTS_DIR=/home/santig14/fing/taa/2_TallerAA/02_preprocesamiento/results


## Construccion de la base comun


In [2]:
def labels_per_row(df):
    return df['labels'].map(lambda raw: len(split_labels(raw)))

summary = pd.DataFrame([
    {'split': 'train_curated', 'rows': len(curated), 'unique_files': curated['fname'].nunique(), 'labels_per_row_mean': labels_per_row(curated).mean()},
    {'split': 'train_noisy', 'rows': len(noisy), 'unique_files': noisy['fname'].nunique(), 'labels_per_row_mean': labels_per_row(noisy).mean()},
    {'split': 'test', 'rows': len(sample), 'unique_files': sample['fname'].nunique(), 'labels_per_row_mean': np.nan},
])

label_counts = {}
for raw in curated['labels']:
    for label in split_labels(raw):
        label_counts[label] = label_counts.get(label, 0) + 1
label_counts_df = pd.Series(label_counts, name='curated_count').sort_values(ascending=False).rename_axis('label').reset_index()

bad = sorted(set(curated['fname']).intersection(CORRUPT_OR_BAD_LABEL_FILES) | set(noisy['fname']).intersection(CORRUPT_OR_BAD_LABEL_FILES))
bad_df = pd.DataFrame({'known_bad_or_corrupt_file': bad})

display(summary.round(4))
display(label_counts_df.head(15))
display(bad_df)

summary.to_csv(RESULTS_DIR / 'C00_base_audio_summary.csv', index=False)
label_counts_df.to_csv(RESULTS_DIR / 'C00_curated_label_counts.csv', index=False)
bad_df.to_csv(RESULTS_DIR / 'C00_known_bad_files.csv', index=False)
print('Artefactos guardados en', RESULTS_DIR)


,split,rows,unique_files,labels_per_row_mean
0,train_curated,4970,4970,1.1573
1,train_noisy,19815,19815,1.2112
2,test,3361,3361,NaN


,label,curated_count
0,Bark,75
1,Raindrop,75
2,Finger_snapping,75
3,Run,75
4,Whispering,75
5,Acoustic_guitar,75
6,Strum,75
7,Hi-hat,75
8,Bass_drum,75
9,Crowd,75


,known_bad_or_corrupt_file
0,1d44b0bd.wav
1,6a1f682a.wav
2,7752cc8a.wav
3,77b925c2.wav
4,c7db12aa.wav
5,f76181c4.wav


Artefactos guardados en /home/santig14/fing/taa/2_TallerAA/02_preprocesamiento/results


## Decision

- `train_curated.csv` queda como base limpia para validacion controlada.
- `train_noisy.csv` queda fuera del flujo default porque la prueba directa degrad? validacion.
- `sample_submission.csv` fija las 80 clases y el formato final.
- Los archivos problematicos quedan registrados para que los pipelines no los ignoren silenciosamente.
